# H02C8b Information Retrieval and Search Engines: RAG Project

Welcome to the notebook companion for the IRSE project. You will find all starter code here. You are encouraged to use this code, as it has been confirmed to work for the RAG pipeline described in the assignment handout. However, you are certainly welcome to make any changes you see fit, provided that your code is written in Python and runs without issue.

**IMPORTANT**: Do not submit a notebook as your final solution. It will not be graded. Refer to assignment handout for more information about the submission format.

**IMPORTANT**: Be mindful of your runtime usage, if working in Colab. At the beginning of every session, navigate to the top menu bar in Colab and select **Runtime > Change runtime type > CPU (Python 3)**. This will ensure that your session runs on CPU and that you do not waste any GPU allocation for the day. GPUs are provided by Google on a limited daily basis, and access is given every 24 hours. It is best that you complete the TF-IDF/search component before loading models and running inference on the GPU runtime.


If you have any questions, feel free to email [Thomas](mailto:thomas.bauwens@kuleuven.be) or [Kushal](mailto:kushaljayesh.tatariya@kuleuven.be).

## RAG for recipe reasoning

We will begin by installing the huggingface `datasets` library for easily loading our data.

In [1]:
!pip install datasets
!pip install "fsspec==2023.1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.0/143.0 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.1.0 which is incompatible.
bigframes 2.39.0 requires fsspec>=2023.3.0, but you have fsspec 2023.1.0 which is incompatible.
huggingface-hub 1.11.0 requires fsspec>=2023.5.0, but you have fsspec 2023.1.0 which is incompatible.


In [2]:
import json
import datasets

Let's first download the recipes dataset. After that, we can load the dataset via the huggingface `datasets` library, which offers easy integration with `transformers`.

In [3]:
!wget https://people.cs.kuleuven.be/~thomas.bauwens/irse_documents_2026_recipes.parquet

--2026-05-16 14:53:25--  https://people.cs.kuleuven.be/~thomas.bauwens/irse_documents_2026_recipes.parquet
Resolving people.cs.kuleuven.be (people.cs.kuleuven.be)... 134.58.40.32, 2a02:2c40:500:a030:c515:1337:0:32
Connecting to people.cs.kuleuven.be (people.cs.kuleuven.be)|134.58.40.32|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 119645533 (114M)
Saving to: ‘irse_documents_2026_recipes.parquet’

irse_documents_2026 100%[===================>] 114.10M  26.4MB/s    in 5.0s    

2026-05-16 14:53:31 (22.8 MB/s) - ‘irse_documents_2026_recipes.parquet’ saved [119645533/119645533]



In [4]:
dataset = datasets.load_dataset("parquet", data_files="./irse_documents_2026_recipes.parquet")['train']

Generating train split: 0 examples [00:00, ? examples/s]

`datasets` allows us to index directly into a `Dataset` object and easily access the data associated with a sample. You can find more information about working with datasets [here](https://huggingface.co/docs/datasets/access).

In [5]:
print("One document:")
for example in dataset:
    for k,v in example.items():
        print(f"'{k}' = {v}\n")
    break

One document:
'name' = arriba baked winter squash mexican style

'ingredients' = winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt

'steps' = make a choice and proceed with recipe, depending on size of squash , cut into half or fourths, remove seeds, for spicy squash , drizzle olive oil or melted butter over each cut squash piece, season with mexican seasoning mix ii, for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece, season with sweet mexican spice mix, bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin, be careful not to burn the squash especially if you opt to use sugar or butter, if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking, if desired , season with salt

'tags' = 60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, preparation, occasion, north-american, side-

We can also load the `queries.json` file, which contains the gold queries created by the instructors. You can use this to debug your retriever and estimate MAP (see project instructions for details).

In [6]:
!wget https://people.cs.kuleuven.be/~thomas.bauwens/irse_queries_2026_recipes.json

--2026-05-16 14:53:32--  https://people.cs.kuleuven.be/~thomas.bauwens/irse_queries_2026_recipes.json
Resolving people.cs.kuleuven.be (people.cs.kuleuven.be)... 134.58.40.32, 2a02:2c40:500:a030:c515:1337:0:32
Connecting to people.cs.kuleuven.be (people.cs.kuleuven.be)|134.58.40.32|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27562 (27K) [application/json]
Saving to: ‘irse_queries_2026_recipes.json’

irse_queries_2026_r 100%[===================>]  26.92K  --.-KB/s    in 0.1s    

2026-05-16 14:53:32 (250 KB/s) - ‘irse_queries_2026_recipes.json’ saved [27562/27562]



In [7]:
queries = json.load(open("./irse_queries_2026_recipes.json", "r"))
print(queries["queries"][0])

{'q': 'What temperature should I pre-heat my oven to when making chicken quesadillas?', 'r': [[167945, 1], [167954, 1], [21548, 1], [218187, 1], [168524, 1], [68174, 1], [34390, 1], [34410, 1], [85623, 1], [46749, 1], [83613, 1], [210101, 1], [192707, 1], [19157, 1], [46809, 1], [168697, 1], [139022, 1], [168732, 1], [151851, 1], [45356, 1], [45357, 1], [40241, 1], [6453, 1], [179511, 1], [168270, 1], [19788, 1], [223067, 1], [19339, 1], [98716, 1], [191431, 1], [25072, 1]], 'a': '375 is a good temperature, but can go as low as 350 or as high as 400. Adjust times accordingly (longer for lower temperatures).'}


You can see that the `queries` dictionary object contains a list of dictionaries, consisting of query (`q`), answer (`a`), and relevant documents (`r`) fields. The integer values in `r` correspond to the `official_id` field in the `recipes.parquet` dataset (see above), along with a relevance score.



Now that the dataset and queries have been loaded, you are free to implement your RAG pipeline. You are welcome to use any implementation of TF-IDF that you are familiar with. Keep in mind that the TF-IDF model must be fit on the recipes dataset provided above, although you are free to experiment what field is most salient for your relevant document search.

In [8]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.8 MB/s eta 0:00:00


In [9]:
"""
retrieval.py
------------
TF-IDF retrieval over the recipes dataset.

Design notes:
- Documents are constructed by concatenating selected fields per recipe.
  Default fields: name + ingredients + tags. (See assignment Task 3 — to be
  validated by ablation experiment.)
- We use sklearn's TfidfVectorizer (defaults: sublinear_tf=False,
  norm='l2', smooth_idf=True). The IDF formula is:
      idf(t) = ln((1 + N) / (1 + df(t))) + 1
  and each document/query vector is L2-normalized after weighting.
- Query vectors use the same vectorizer (fit on the corpus), so OOV
  query terms are simply dropped. A fully-OOV query yields a zero vector,
  which we detect and report.
- Similarity: cosine. Because vectors are L2-normalized, cosine == dot product.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Iterable

import numpy as np
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer


# --- Field handling ---------------------------------------------------------

def _normalize_field(value) -> str:
    """Coerce any dataset field into a single space-joined string.

    Recipes fields may be strings, list-of-strings (ingredients, steps, tags),
    or occasionally None. We flatten all of them to plain text so the
    vectorizer can tokenize uniformly.
    """
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, (list, tuple)):
        return " ".join(_normalize_field(v) for v in value)
    return str(value)


def build_document_text(recipe: dict, fields: Iterable[str]) -> str:
    """Concatenate selected fields of a recipe into one string.

    Tags often contain hyphens (e.g. '60-minutes-or-less'). Hyphens are
    replaced with spaces so the tokenizer sees the constituent words.
    """
    parts = []
    for f in fields:
        text = _normalize_field(recipe.get(f))
        if f == "tags":
            text = text.replace("-", " ")
        if text:
            parts.append(text)
    return " ".join(parts)


# --- Retriever --------------------------------------------------------------

DEFAULT_FIELDS = ("name", "ingredients", "tags")


@dataclass
class RetrievalResult:
    """One retrieval result: dataset index, recipe id, similarity score."""
    dataset_index: int
    official_id: int | None
    score: float


@dataclass
class TfidfRetriever:
    """TF-IDF retriever with cosine similarity over a recipe corpus.

    Usage:
        r = TfidfRetriever(fields=("name", "ingredients", "tags"))
        r.fit(dataset)                       # dataset = HF Dataset or list of dicts
        hits = r.retrieve("shrimp tacos", k=10)

    Optional MWE support:
        from mwe import MWEDetector
        mwe = MWEDetector().fit(doc_texts)
        r = TfidfRetriever(fields=..., mwe=mwe)
        r.fit(dataset)
        # Now both documents and queries get MWE-merged before vectorization.
    """

    fields: tuple[str, ...] = DEFAULT_FIELDS

    # Vectorizer hyperparameters — exposed so experiments can vary them
    # without subclassing.
    lowercase: bool = True
    min_df: int = 2          # drop hapax legomena; reduces vocab + noise
    max_df: float = 0.95     # drop terms in >95% of docs (very weak signal)
    ngram_range: tuple[int, int] = (1, 1)   # bumped to (1,2) when MWEs enabled
    stop_words: str | None = "english"
    token_pattern: str = r"(?u)\b[a-z][a-z_]+\b"  # alphabetic OR underscore (for MWEs), length >= 2

    # Optional MWE detector. If provided, .fit() and .encode_query() apply
    # the detector before handing text to the vectorizer.
    mwe: "MWEDetector | None" = None

    # Populated by fit()
    vectorizer: TfidfVectorizer | None = field(default=None, init=False)
    doc_matrix: csr_matrix | None = field(default=None, init=False)
    official_ids: list[int] | None = field(default=None, init=False)

    # ---- fit / index ----

    def fit(self, dataset) -> "TfidfRetriever":
        """Build the term vocabulary and document-term matrix.

        `dataset` can be a HuggingFace Dataset, a list of dicts, or anything
        that iterates over recipe dicts containing at least `self.fields`.
        """
        texts = []
        official_ids = []
        for recipe in dataset:
            texts.append(build_document_text(recipe, self.fields))
            official_ids.append(recipe.get("official_id"))

        # If an MWE detector is attached, merge collocations in each document
        # before fitting the vectorizer. "olive oil" -> "olive_oil".
        if self.mwe is not None:
            texts = [self.mwe.transform_text(t) for t in texts]

        self.vectorizer = TfidfVectorizer(
            lowercase=self.lowercase,
            min_df=self.min_df,
            max_df=self.max_df,
            ngram_range=self.ngram_range,
            stop_words=self.stop_words,
            token_pattern=self.token_pattern,
            sublinear_tf=False,
            norm="l2",
            smooth_idf=True,
        )
        self.doc_matrix = self.vectorizer.fit_transform(texts)
        self.official_ids = official_ids
        return self

    # ---- query ----

    def encode_query(self, query: str) -> csr_matrix:
        """Turn a query string into a TF-IDF row vector using the fitted vocab.

        OOV terms are silently dropped (sklearn behavior). If *all* terms are
        OOV, the returned vector has zero nnz — callers should check for this.
        """
        if self.vectorizer is None:
            raise RuntimeError("Retriever not fit; call .fit(dataset) first.")
        # Apply the same MWE transform to queries that we applied to docs,
        # so a query for "olive oil" matches against the merged "olive_oil"
        # token in the vocabulary.
        if self.mwe is not None:
            query = self.mwe.transform_text(query)
        return self.vectorizer.transform([query])

    def is_query_in_vocab(self, query: str) -> bool:
        """True iff the query has at least one in-vocabulary token."""
        return self.encode_query(query).nnz > 0

    def retrieve(self, query: str, k: int = 10) -> list[RetrievalResult]:
        """Return top-k recipes by cosine similarity.

        Vectors are L2-normalized, so cosine = dot product. We compute the
        sparse-dense product, then partial-sort for the top-k.
        """
        if self.doc_matrix is None:
            raise RuntimeError("Retriever not fit; call .fit(dataset) first.")

        q_vec = self.encode_query(query)
        # (1 x V) @ (V x N).T -> (1 x N) dense scores
        scores = (self.doc_matrix @ q_vec.T).toarray().ravel()

        if k >= len(scores):
            top_idx = np.argsort(-scores)
        else:
            # argpartition is O(N); sort only the top-k slice
            top_idx = np.argpartition(-scores, k)[:k]
            top_idx = top_idx[np.argsort(-scores[top_idx])]

        results = []
        for idx in top_idx:
            results.append(RetrievalResult(
                dataset_index=int(idx),
                official_id=self.official_ids[idx],
                score=float(scores[idx]),
            ))
        return results

In [10]:
"""
mwe.py
------
Multi-word expression (MWE) detection for the recipes corpus.

We use gensim's `Phrases` with NPMI (Normalized Pointwise Mutual
Information) scoring:

    PMI(w1, w2)  = log( P(w1 w2) / (P(w1) * P(w2)) )
    NPMI(w1, w2) = PMI(w1, w2) / -log( P(w1 w2) )

NPMI is bounded in [-1, 1], where:
    -1 = w1 and w2 never co-occur
     0 = independent (no association)
    +1 = w1 and w2 always co-occur (one fully predicts the other)

We keep bigrams with NPMI above `threshold` (default 0.5: strong
collocation). Applying the model twice allows longer MWEs to form
(e.g. "extra_virgin_olive_oil" from "extra_virgin" + "olive_oil").

Integration with sklearn TfidfVectorizer:
- `transform_text(text) -> str` returns a string with detected MWEs
  joined by underscores.
- The retriever applies this transform to every document before fitting
  and to every query before transform. sklearn's `token_pattern` is
  configured to accept underscores so the merged tokens survive
  re-tokenization.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Iterable

from gensim.models.phrases import Phrases


# Match the retriever's tokenizer: lowercase alphabetic tokens of length >= 2.
# Keeping these aligned matters: gensim must see the same tokens that
# sklearn will eventually see after merging.
_TOKEN_RE = re.compile(r"\b[a-z][a-z]+\b")


def _tokenize(text: str) -> list[str]:
    """Same tokenization as TfidfRetriever's default token_pattern."""
    return _TOKEN_RE.findall(text.lower())


@dataclass
class MWEDetector:
    """Two-pass NPMI-based MWE detection via gensim Phrases.

    Two passes:
      pass 1: bigrams        (olive + oil -> olive_oil)
      pass 2: bigrams again  (extra_virgin + olive_oil -> extra_virgin_olive_oil)

    Parameters:
      min_count: ignore bigrams whose total count is below this.
                 Higher = fewer, more confident MWEs.
      threshold: NPMI score threshold in [-1, 1]. Higher = stricter.
                 0.5 is a reasonable default for "clearly collocated".
    """

    min_count: int = 20
    threshold: float = 0.5
    two_passes: bool = True

    bigram_model: Phrases | None = field(default=None, init=False)
    trigram_model: Phrases | None = field(default=None, init=False)

    # ---- fit ----

    def fit(self, texts: Iterable[str]) -> "MWEDetector":
        """Train the Phrases model(s) on a corpus of documents.

        `texts` is an iterable of pre-built document strings (e.g. the same
        strings the retriever feeds to sklearn).
        """
        tokenized = [_tokenize(t) for t in texts]

        self.bigram_model = Phrases(
            tokenized,
            min_count=self.min_count,
            threshold=self.threshold,
            delimiter="_",
            scoring="npmi",
        )

        if self.two_passes:
            # Apply the bigram model to the corpus, then train a second
            # pass on the merged tokens. Already-merged "olive_oil" can now
            # combine with another token (e.g. "virgin_olive_oil").
            merged = [list(self.bigram_model[toks]) for toks in tokenized]
            self.trigram_model = Phrases(
                merged,
                min_count=self.min_count,
                threshold=self.threshold,
                delimiter="_",
                scoring="npmi",
            )

        return self

    # ---- apply ----

    def transform_tokens(self, tokens: list[str]) -> list[str]:
        """Apply the trained model(s) to a list of tokens."""
        if self.bigram_model is None:
            raise RuntimeError("MWEDetector not fit; call .fit(texts) first.")
        out = list(self.bigram_model[tokens])
        if self.two_passes and self.trigram_model is not None:
            out = list(self.trigram_model[out])
        return out

    def transform_text(self, text: str) -> str:
        """Tokenize, merge MWEs, and rejoin as a space-separated string.

        The output is suitable for direct feeding to TfidfVectorizer.
        """
        tokens = _tokenize(text)
        merged = self.transform_tokens(tokens)
        return " ".join(merged)

    # ---- inspection ----

    def inspect_top_mwes(self, n: int = 30) -> list[tuple[str, float]]:
        """Return the top-n highest-scoring detected MWEs (bigrams only).

        Useful for sanity-checking what got merged.
        """
        if self.bigram_model is None:
            raise RuntimeError("MWEDetector not fit; call .fit(texts) first.")
        phrases = dict(self.bigram_model.export_phrases())
        items = sorted(phrases.items(), key=lambda kv: -kv[1])
        return items[:n]

In [11]:
retrieval_queries = [
    "cajun style gumbo with an easy roux",
    "shrimp tacos recipe for tonight",
    "vegetarian lasagna but easy",
    "spageti and meatballs?",
    "lunch baked at 200 °C"
]

In [12]:
# Fit the retriever on the recipes corpus
r = TfidfRetriever(fields=("name", "ingredients", "tags", "steps")).fit(dataset)
print(f"Vocab size: {len(r.vectorizer.vocabulary_)}")
print(f"Doc matrix shape: {r.doc_matrix.shape}")
print()

# Try the 5 example queries from the notebook
for q in retrieval_queries:
    print(f"Query: {q!r}")
    print(f"  In-vocab: {r.is_query_in_vocab(q)}")
    hits = r.retrieve(q, k=3)
    for h in hits:
        recipe = dataset[h.dataset_index]
        print(f"    id={h.official_id} score={h.score:.4f} -> {recipe['name']}")
    print()

Vocab size: 27642
Doc matrix shape: (231637, 27642)

Query: 'cajun style gumbo with an easy roux'
  In-vocab: True
    id=73182 score=0.6786 -> double roux cajun seafood gumbo
    id=76648 score=0.5391 -> easy gumbo
    id=186726 score=0.5046 -> simple chicken gumbo

Query: 'shrimp tacos recipe for tonight'
  In-vocab: True
    id=211508 score=0.4198 -> best garlic shrimp whole wide world
    id=187265 score=0.3696 -> simple shrimp tacos
    id=185758 score=0.3447 -> shrimp gazpacho

Query: 'vegetarian lasagna but easy'
  In-vocab: True
    id=119738 score=0.6920 -> lasagna pierogi
    id=64585 score=0.6831 -> crisp lasagna chips
    id=6883 score=0.6449 -> apple lasagna

Query: 'spageti and meatballs?'
  In-vocab: True
    id=131244 score=0.7877 -> marmalade meatballs
    id=10506 score=0.7744 -> aunt verna meatballs
    id=507 score=0.7608 -> please make meatballs crockpot meatballs

Query: 'lunch baked at 200 °C'
  In-vocab: True
    id=220408 score=0.4446 -> vancouver baked bean sa

In [13]:
# --- Diagnostic 1: vocab check for the typo case ---
print("=== Vocab check ===")
for w in ["spageti", "spaghetti", "meatballs", "lunch", "baked", "tonight"]:
    print(f"  {w!r:15s} in vocab: {w in r.vectorizer.vocabulary_}")
print()

# --- Diagnostic 2: what tokens actually survive for each query? ---
print("=== Surviving tokens per query (after tokenization + vocab filter) ===")
analyzer = r.vectorizer.build_analyzer()  # same pipeline used at fit time
for q in retrieval_queries:
    raw_tokens = analyzer(q)                       # pre-vocab tokenization
    in_vocab = [t for t in raw_tokens if t in r.vectorizer.vocabulary_]
    dropped  = [t for t in raw_tokens if t not in r.vectorizer.vocabulary_]
    print(f"  query: {q!r}")
    print(f"    tokens after tokenization: {raw_tokens}")
    print(f"    survived (in vocab):       {in_vocab}")
    print(f"    dropped (OOV / filtered):  {dropped}")
    print()

=== Vocab check ===
  'spageti'       in vocab: False
  'spaghetti'     in vocab: True
  'meatballs'     in vocab: True
  'lunch'         in vocab: True
  'baked'         in vocab: True
  'tonight'       in vocab: True

=== Surviving tokens per query (after tokenization + vocab filter) ===
  query: 'cajun style gumbo with an easy roux'
    tokens after tokenization: ['cajun', 'style', 'gumbo', 'easy', 'roux']
    survived (in vocab):       ['cajun', 'style', 'gumbo', 'easy', 'roux']
    dropped (OOV / filtered):  []

  query: 'shrimp tacos recipe for tonight'
    tokens after tokenization: ['shrimp', 'tacos', 'recipe', 'tonight']
    survived (in vocab):       ['shrimp', 'tacos', 'recipe', 'tonight']
    dropped (OOV / filtered):  []

  query: 'vegetarian lasagna but easy'
    tokens after tokenization: ['vegetarian', 'lasagna', 'easy']
    survived (in vocab):       ['vegetarian', 'lasagna', 'easy']
    dropped (OOV / filtered):  []

  query: 'spageti and meatballs?'
    tokens after 

Once your TF-IDF model has been implemented and fit on the recipes dataset, you can experiment with retrieving the k-most relevant documents for the document-like queries provided below:

In [14]:
"""
evaluation.py
-------------
Evaluate an IR retriever against a gold-query set.

Metrics (binary relevance — see assignment Task 4):
- Per-query precision, recall, F1 at fixed k.
- Macro-average: mean of per-query metrics.
- Micro-average: pool TP/FP/FN across all queries, then compute.
- Mean Average Precision (MAP): mean over queries of Average Precision (AP).

Average Precision formula (for binary relevance):
    AP(q) = (1 / |R_q|) * sum_{k=1..K} [ P@k(q) * rel(k) ]
where:
    R_q       = set of all relevant documents for query q
    K         = number of retrieved documents (the system's chosen k)
    P@k(q)    = precision considering only the top-k retrieved docs
    rel(k)    = 1 if the k-th retrieved doc is relevant, else 0

Then MAP = (1/|Q|) * sum_{q in Q} AP(q).

Note on the "at k" question: AP here is computed *over the system's
retrieved list of size K*, not over the full corpus. This is the standard
formulation when the system commits to a fixed retrieval cutoff. If the
system retrieves fewer than |R_q| documents, AP is bounded above by K/|R_q|.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable

import numpy as np


# --- Mapping: official_id <-> dataset index --------------------------------

def build_id_to_index(dataset) -> dict[int, int]:
    """Map each recipe's official_id to its position in the dataset.

    The gold queries reference official_id; the retriever returns dataset
    positions. We need to translate between them in both directions.
    """
    mapping = {}
    for i, recipe in enumerate(dataset):
        oid = recipe.get("official_id")
        if oid is not None:
            mapping[oid] = i
    return mapping


# --- Per-query metrics -----------------------------------------------------

@dataclass
class QueryEval:
    """Per-query evaluation result."""
    query: str
    retrieved_ids: list[int]       # official_ids returned by the system, in rank order
    relevant_ids: set[int]         # official_ids marked relevant in gold
    tp: int                        # # retrieved AND relevant
    fp: int                        # # retrieved AND NOT relevant
    fn: int                        # # NOT retrieved BUT relevant
    precision: float
    recall: float
    f1: float
    average_precision: float


def _f1(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def evaluate_query(query: str,
                   retrieved_ids: list[int],
                   relevant_ids: Iterable[int]) -> QueryEval:
    """Compute precision/recall/F1/AP for a single query.

    Args:
        query:         The query string (stored for inspection).
        retrieved_ids: Official IDs returned by the system, in rank order.
        relevant_ids:  Official IDs that are gold-relevant (any score >= 1).
    """
    relevant_set = set(relevant_ids)
    retrieved_set = set(retrieved_ids)

    tp = len(retrieved_set & relevant_set)
    fp = len(retrieved_set - relevant_set)
    fn = len(relevant_set - retrieved_set)

    precision = tp / len(retrieved_set) if retrieved_set else 0.0
    recall = tp / len(relevant_set) if relevant_set else 0.0
    f1 = _f1(precision, recall)

    # --- Average Precision (rank-aware) ---
    # Walk down the retrieved list; at each position where the doc is
    # relevant, record the running precision; average those values over
    # the total count of relevant docs.
    ap_sum = 0.0
    hits_so_far = 0
    for k, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_set:
            hits_so_far += 1
            precision_at_k = hits_so_far / k
            ap_sum += precision_at_k
    ap = ap_sum / len(relevant_set) if relevant_set else 0.0

    return QueryEval(
        query=query,
        retrieved_ids=list(retrieved_ids),
        relevant_ids=relevant_set,
        tp=tp, fp=fp, fn=fn,
        precision=precision, recall=recall, f1=f1,
        average_precision=ap,
    )


# --- Aggregate metrics over a query set ------------------------------------

@dataclass
class EvalReport:
    """Summary statistics across a query set."""
    n_queries: int
    k: int

    # Macro: mean of per-query metrics
    macro_precision: float
    macro_recall: float
    macro_f1: float

    # Micro: pooled counts then compute
    micro_precision: float
    micro_recall: float
    micro_f1: float

    # Ranking metric
    mean_average_precision: float

    # Keep per-query results for drill-down
    per_query: list[QueryEval]

    def pretty(self) -> str:
        lines = [
            f"Evaluation report  (N={self.n_queries} queries, k={self.k})",
            "-" * 50,
            f"  Macro-avg precision: {self.macro_precision:.4f}",
            f"  Macro-avg recall:    {self.macro_recall:.4f}",
            f"  Macro-avg F1:        {self.macro_f1:.4f}",
            "",
            f"  Micro-avg precision: {self.micro_precision:.4f}",
            f"  Micro-avg recall:    {self.micro_recall:.4f}",
            f"  Micro-avg F1:        {self.micro_f1:.4f}",
            "",
            f"  MAP:                 {self.mean_average_precision:.4f}",
        ]
        return "\n".join(lines)


def evaluate_retriever(retriever, queries: list[dict], k: int) -> EvalReport:
    """Run the retriever over a gold-query set and compute all metrics.

    Args:
        retriever:  An object with a `.retrieve(query, k)` method returning
                    objects that have an `.official_id` attribute (matches
                    TfidfRetriever's RetrievalResult).
        queries:    List of dicts with keys 'q' (query string) and
                    'r' (list of [official_id, relevance_score] pairs).
        k:          Fixed retrieval cutoff for the system.

    Returns:
        EvalReport with macro, micro, and MAP metrics.
    """
    per_query = []
    total_tp = total_fp = total_fn = 0

    for q in queries:
        query_text = q["q"]
        relevant_ids = [pair[0] for pair in q["r"]]  # drop scores, binary relevance

        hits = retriever.retrieve(query_text, k=k)
        retrieved_ids = [h.official_id for h in hits if h.official_id is not None]

        eval_q = evaluate_query(query_text, retrieved_ids, relevant_ids)
        per_query.append(eval_q)

        total_tp += eval_q.tp
        total_fp += eval_q.fp
        total_fn += eval_q.fn

    # Macro: mean over per-query metrics
    macro_p = float(np.mean([q.precision for q in per_query]))
    macro_r = float(np.mean([q.recall for q in per_query]))
    macro_f1 = float(np.mean([q.f1 for q in per_query]))

    # Micro: pool counts then compute
    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = _f1(micro_p, micro_r)

    mean_ap = float(np.mean([q.average_precision for q in per_query]))

    return EvalReport(
        n_queries=len(queries),
        k=k,
        macro_precision=macro_p,
        macro_recall=macro_r,
        macro_f1=macro_f1,
        micro_precision=micro_p,
        micro_recall=micro_r,
        micro_f1=micro_f1,
        mean_average_precision=mean_ap,
        per_query=per_query,
    )


# --- Convenience: scan over multiple k values ------------------------------

def evaluate_at_multiple_k(retriever, queries: list[dict],
                           ks: Iterable[int]) -> dict[int, EvalReport]:
    """Run evaluation for several values of k. Useful for k-sweeps."""
    return {k: evaluate_retriever(retriever, queries, k=k) for k in ks}

In [15]:
from gensim.models.phrases import Phrases  # ensure gensim importable
import time

# Step 1: build document texts (need them as plain strings to train MWE detector)
print("Building document texts...")
doc_texts = [build_document_text(rec, ("name", "ingredients", "tags", "steps"))
             for rec in dataset]
print(f"  {len(doc_texts)} documents")

# Step 2: train the MWE detector
print("\nTraining MWE detector (this takes 1-3 min on 231k docs)...")
t0 = time.time()
mwe = MWEDetector(min_count=30, threshold=0.5).fit(doc_texts)
print(f"  done in {time.time()-t0:.1f}s")

# Step 3: inspect what got detected
print("\nTop 30 detected MWEs by NPMI:")
for phrase, score in mwe.inspect_top_mwes(n=30):
    print(f"  {phrase:<30s} NPMI={score:.3f}")

# Step 4: fit a retriever using the MWE detector and the +steps fields
print("\nFitting MWE-aware retriever...")
t0 = time.time()
r_mwe = TfidfRetriever(
    fields=("name", "ingredients", "tags", "steps"),
    mwe=mwe
).fit(dataset)
print(f"  done in {time.time()-t0:.1f}s")
print(f"  vocab size: {len(r_mwe.vectorizer.vocabulary_)}")

# Step 5: evaluate
print("\nEvaluating...")
rep_mwe = evaluate_retriever(r_mwe, queries["queries"], k=10)
print(rep_mwe.pretty())

# Step 6: compare to no-MWE baseline (+steps fields only)
print("\n=== Comparison ===")
print(f"  no MWE  (+steps):  MAP=0.219, F1=0.214")
print(f"  with MWE (+steps): MAP={rep_mwe.mean_average_precision:.3f}, "
      f"F1={rep_mwe.macro_f1:.3f}")

Building document texts...
  231637 documents

Training MWE detector (this takes 1-3 min on 231k docs)...
  done in 119.3s

Top 30 detected MWEs by NPMI:
  united_states                  NPMI=1.000
  tex_mex                        NPMI=0.999
  duncan_hines                   NPMI=0.999
  holiday_event                  NPMI=0.998
  heirloom_historical            NPMI=0.998
  barefoot_contessa              NPMI=0.997
  beau_monde                     NPMI=0.993
  grand_marnier                  NPMI=0.993
  mardi_gras                     NPMI=0.992
  ro_tel                         NPMI=0.992
  british_columbian              NPMI=0.990
  ping_pong                      NPMI=0.990
  todd_wilbur                    NPMI=0.989
  rosh_hashana                   NPMI=0.989
  dinner_party                   NPMI=0.988
  panna_cotta                    NPMI=0.987
  lea_perrins                    NPMI=0.987
  granny_smith                   NPMI=0.986
  al_dente                       NPMI=0.986
  rachael_

In [16]:
import time

# --- MWE v2: exclude description from training, raise min_count ---
print("Building MWE training texts (no description)...")
mwe_texts = [build_document_text(rec, ("name", "ingredients", "tags", "steps"))
             for rec in dataset]

print("\nTraining MWE detector v2 (min_count=200, no description)...")
t0 = time.time()
mwe_v2 = MWEDetector(min_count=200, threshold=0.5).fit(mwe_texts)
print(f"  done in {time.time()-t0:.1f}s")

print("\nTop 30 detected MWEs by NPMI:")
for phrase, score in mwe_v2.inspect_top_mwes(n=30):
    print(f"  {phrase:<30s} NPMI={score:.3f}")

print("\nFitting MWE-aware retriever v2...")
t0 = time.time()
r_mwe_v2 = TfidfRetriever(
    fields=("name", "ingredients", "tags", "steps"),
    mwe=mwe_v2
).fit(dataset)
print(f"  done in {time.time()-t0:.1f}s")
print(f"  vocab size: {len(r_mwe_v2.vectorizer.vocabulary_)}")

print("\nEvaluating...")
rep_mwe_v2 = evaluate_retriever(r_mwe_v2, queries["queries"], k=10)
print(rep_mwe_v2.pretty())

print("\n=== Comparison ===")
print(f"  no MWE  (+steps):  MAP=0.219, F1=0.214")
print(f"  MWE v1 (NPMI=0.5, min=30, all 4 fields): MAP=0.219, F1=0.211")
print(f"  MWE v2 (NPMI=0.5, min=200, no desc):     "
      f"MAP={rep_mwe_v2.mean_average_precision:.3f}, "
      f"F1={rep_mwe_v2.macro_f1:.3f}")

Building MWE training texts (no description)...

Training MWE detector v2 (min_count=200, no description)...
  done in 117.9s

Top 30 detected MWEs by NPMI:
  united_states                  NPMI=1.000
  tex_mex                        NPMI=0.999
  holiday_event                  NPMI=0.998
  heirloom_historical            NPMI=0.998
  grand_marnier                  NPMI=0.993
  mardi_gras                     NPMI=0.992
  british_columbian              NPMI=0.990
  todd_wilbur                    NPMI=0.989
  rosh_hashana                   NPMI=0.989
  dinner_party                   NPMI=0.988
  granny_smith                   NPMI=0.986
  al_dente                       NPMI=0.986
  rachael_ray                    NPMI=0.985
  gras_carnival                  NPMI=0.985
  hidden_valley                  NPMI=0.984
  bok_choy                       NPMI=0.984
  condiments_etc                 NPMI=0.983
  paula_deen                     NPMI=0.983
  st_patricks                    NPMI=0.983
  kid_f

In [17]:
# Run evaluation over the gold queries at k=10
report = evaluate_retriever(r, queries["queries"], k=10)
print(report.pretty())

# Quick k-sweep to find a reasonable cutoff
print("\n=== K-sweep ===")
for k in [5, 10, 20, 50, 100]:
    rep = evaluate_retriever(r, queries["queries"], k=k)
    print(f"  k={k:3d}  macro_P={rep.macro_precision:.3f}  macro_R={rep.macro_recall:.3f}  "
          f"macro_F1={rep.macro_f1:.3f}  MAP={rep.mean_average_precision:.3f}")

Evaluation report  (N=47 queries, k=10)
--------------------------------------------------
  Macro-avg precision: 0.2383
  Macro-avg recall:    0.2798
  Macro-avg F1:        0.2140

  Micro-avg precision: 0.2383
  Micro-avg recall:    0.1934
  Micro-avg F1:        0.2135

  MAP:                 0.2188

=== K-sweep ===
  k=  5  macro_P=0.298  macro_R=0.214  macro_F1=0.211  MAP=0.182
  k= 10  macro_P=0.238  macro_R=0.280  macro_F1=0.214  MAP=0.219
  k= 20  macro_P=0.178  macro_R=0.366  macro_F1=0.199  MAP=0.247
  k= 50  macro_P=0.117  macro_R=0.515  macro_F1=0.163  MAP=0.278
  k=100  macro_P=0.075  macro_R=0.587  macro_F1=0.116  MAP=0.291


In [18]:
# ---- Field ablation experiment (Task 3 requirement) ----
# We measure how MAP / macro-F1 change with different field choices.

field_configs = {
    "name+ing+tags":        ("name", "ingredients", "tags"),
    "+steps":               ("name", "ingredients", "tags", "steps"),
    "+description":         ("name", "ingredients", "tags", "description"),
    "all 5 fields":         ("name", "ingredients", "tags", "steps", "description"),
    "name+ing only":        ("name", "ingredients"),
}

results = {}
for label, fields in field_configs.items():
    print(f"\n=== Fitting: {label}  fields={fields} ===")
    retr = TfidfRetriever(fields=fields).fit(dataset)
    print(f"    vocab size = {len(retr.vectorizer.vocabulary_)}")
    rep = evaluate_retriever(retr, queries["queries"], k=10)
    results[label] = (rep, retr)
    print(f"    MAP={rep.mean_average_precision:.3f}  "
          f"macro_F1={rep.macro_f1:.3f}  "
          f"macro_P={rep.macro_precision:.3f}  "
          f"macro_R={rep.macro_recall:.3f}")

# Summary table
print("\n" + "=" * 70)
print(f"{'config':<20s}  {'vocab':>7s}  {'MAP':>6s}  {'F1':>6s}  {'P':>6s}  {'R':>6s}")
print("-" * 70)
for label, (rep, retr) in results.items():
    vocab_size = len(retr.vectorizer.vocabulary_)
    print(f"{label:<20s}  {vocab_size:>7d}  "
          f"{rep.mean_average_precision:>6.3f}  {rep.macro_f1:>6.3f}  "
          f"{rep.macro_precision:>6.3f}  {rep.macro_recall:>6.3f}")


=== Fitting: name+ing+tags  fields=('name', 'ingredients', 'tags') ===
    vocab size = 13786
    MAP=0.213  macro_F1=0.213  macro_P=0.234  macro_R=0.290

=== Fitting: +steps  fields=('name', 'ingredients', 'tags', 'steps') ===
    vocab size = 27642
    MAP=0.219  macro_F1=0.214  macro_P=0.238  macro_R=0.280

=== Fitting: +description  fields=('name', 'ingredients', 'tags', 'description') ===
    vocab size = 36722
    MAP=0.172  macro_F1=0.240  macro_P=0.272  macro_R=0.308

=== Fitting: all 5 fields  fields=('name', 'ingredients', 'tags', 'steps', 'description') ===
    vocab size = 44364
    MAP=0.182  macro_F1=0.224  macro_P=0.257  macro_R=0.287

=== Fitting: name+ing only  fields=('name', 'ingredients') ===
    vocab size = 13754
    MAP=0.197  macro_F1=0.195  macro_P=0.200  macro_R=0.277

config                  vocab     MAP      F1       P       R
----------------------------------------------------------------------
name+ing+tags           13786   0.213   0.213   0.234   0.29

In [19]:
quesadilla_retr = results["+steps"][1]
hits = quesadilla_retr.retrieve(
    "What temperature should I pre-heat my oven to when making chicken quesadillas?",
    k=10
)
for h in hits:
    print(f"  id={h.official_id} score={h.score:.4f} -> {dataset[h.dataset_index]['name']}")

  id=157704 score=0.4183 -> pepper jack chicken peach quesadillas
  id=24116 score=0.4083 -> black bean chicken quesadillas
  id=99653 score=0.3751 -> grilled quesadillas feta spinach olive lemon relish
  id=7046 score=0.3733 -> apple pie quesadillas
  id=228843 score=0.3724 -> ww weight watchers chicken cheese quesadillas
  id=168697 score=0.3597 -> quick caribbean quesadillas
  id=24101 score=0.3582 -> black bean avocado quesadillas
  id=42777 score=0.3577 -> chicken jalapeno quesadillas nuwave oven flavorwave
  id=231477 score=0.3548 -> zucchini corn black bean jack cheese quesadillas
  id=167959 score=0.3525 -> quesadillas smoked gouda caramelized onion


For a given query and set of relevant documents, you are also required to create a prompt that instructs a model to complete a certain task. You should experiment with formatting the prompt, as language models have been shown to be sensitive to the exact verbiage of instructions.

In [20]:
"""
generation.py
-------------
LM generation for the RAG pipeline.

Two prompt templates are provided (Task 6 requirement):

Prompt A ("Structured"): Gives the LM a clear role, numbered recipes with
    specific fields, and explicit instructions to base its answer on the
    provided context only.

Prompt B ("Minimal"): Gives the LM the raw recipe text and a short
    instruction. Less guidance, more freedom — useful for comparing whether
    the LM benefits from structure.

Both use Mistral-v0.2's chat template via `tokenizer.apply_chat_template()`.
Mistral v0.2 does NOT natively support a system role, so we prepend system
instructions to the first user message.

Dataset fields in the prompt (Task 6 requirement):
    We include `name`, `ingredients`, and `steps` in the prompt context.
    Rationale: `name` identifies the recipe; `ingredients` lets the LM
    reason about substitutions/allergies; `steps` lets it answer procedure
    and temperature questions. We exclude `tags` (redundant with name) and
    `description` (chatty noise). This is experimentally validated by
    comparing prompt A vs prompt B and by the irrelevant-context test.
"""

from __future__ import annotations

from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from retrieval import TfidfRetriever, RetrievalResult


# --------------------------------------------------------------------------
# Prompt template A: Structured
# --------------------------------------------------------------------------

SYSTEM_A = """\
You are a helpful cooking assistant. You answer questions about recipes \
based ONLY on the recipe context provided below. If the context does not \
contain enough information to answer the question, say so explicitly. \
Do not use any knowledge beyond the provided recipes."""

def _format_recipe(rank: int, recipe: dict, score: float | None = None) -> str:
    """Format one retrieved recipe for inclusion in the prompt."""
    parts = [f"Recipe {rank}: {recipe['name']}"]
    if score is not None:
        parts[0] += f"  (relevance score: {score:.3f})"
    parts.append(f"  Ingredients: {recipe['ingredients']}")
    parts.append(f"  Steps: {recipe['steps']}")
    return "\n".join(parts)


def build_prompt_a(query: str,
                   recipes: list[dict],
                   scores: list[float] | None = None,
                   include_scores: bool = False) -> list[dict]:
    """Build Prompt A (structured) as a messages list for apply_chat_template.

    Returns a list of {"role": ..., "content": ...} dicts. The system
    message is prepended to the user message (Mistral v0.2 doesn't support
    system role natively).
    """
    context_parts = []
    for i, recipe in enumerate(recipes, 1):
        s = scores[i - 1] if (include_scores and scores) else None
        context_parts.append(_format_recipe(i, recipe, score=s))
    context_block = "\n\n".join(context_parts)

    user_content = f"""{SYSTEM_A}

--- RECIPE CONTEXT ---
{context_block}
--- END CONTEXT ---

Question: {query}

Answer the question based on the recipes above."""

    return [{"role": "user", "content": user_content}]


# --------------------------------------------------------------------------
# Prompt template B: Minimal
# --------------------------------------------------------------------------

SYSTEM_B = """\
Answer the following cooking question using only the recipes provided."""

def build_prompt_b(query: str,
                   recipes: list[dict],
                   scores: list[float] | None = None,
                   include_scores: bool = False) -> list[dict]:
    """Build Prompt B (minimal) as a messages list."""
    recipe_texts = []
    for i, recipe in enumerate(recipes, 1):
        text = f"[{i}] {recipe['name']}: {recipe['ingredients']}. {recipe['steps']}"
        if include_scores and scores:
            text = f"[{i}] (score {scores[i-1]:.3f}) {recipe['name']}: {recipe['ingredients']}. {recipe['steps']}"
        recipe_texts.append(text)
    context = "\n".join(recipe_texts)

    user_content = f"""{SYSTEM_B}

Recipes:
{context}

Question: {query}"""

    return [{"role": "user", "content": user_content}]


# --------------------------------------------------------------------------
# Full RAG pipeline: query -> retrieve -> prompt -> generate
# --------------------------------------------------------------------------

def rag_generate(query: str,
                 retriever: "TfidfRetriever",
                 dataset,
                 tokenizer,
                 model,
                 k: int = 5,
                 prompt_fn=None,
                 include_scores: bool = False,
                 max_new_tokens: int = 512,
                 do_sample: bool = True,
                 temperature: float = 0.7) -> dict:
    """End-to-end RAG: retrieve, format prompt, generate.

    Args:
        query:          Natural language question.
        retriever:      Fitted TfidfRetriever.
        dataset:        HF Dataset (for looking up recipe fields).
        tokenizer:      HuggingFace tokenizer for the LM.
        model:          HuggingFace model (already on GPU).
        k:              Number of documents to retrieve.
        prompt_fn:      Prompt builder (default: build_prompt_a).
        include_scores: Whether to include retrieval scores in the prompt.
        max_new_tokens: Generation length limit.
        do_sample:      Sampling vs greedy decoding.
        temperature:    Sampling temperature (only if do_sample=True).

    Returns:
        Dict with keys: 'query', 'hits', 'prompt_text', 'response'.
    """
    import torch

    if prompt_fn is None:
        prompt_fn = build_prompt_a

    # 1. Retrieve
    hits = retriever.retrieve(query, k=k)
    recipes = [dataset[h.dataset_index] for h in hits]
    scores = [h.score for h in hits]

    # 2. Build prompt
    messages = prompt_fn(query, recipes, scores=scores,
                         include_scores=include_scores)

    # 3. Tokenize using Mistral's chat template
    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=False
    ).to(model.device)

    # 4. Generate
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        output_ids = model.generate(encoded, **gen_kwargs)

    # 5. Decode (skip the input tokens to get only the generated part)
    generated_ids = output_ids[0, encoded.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Also decode the full prompt for inspection
    prompt_text = tokenizer.decode(encoded[0], skip_special_tokens=True)

    return {
        "query": query,
        "hits": hits,
        "recipes": recipes,
        "scores": scores,
        "prompt_text": prompt_text,
        "response": response.strip(),
    }


# --------------------------------------------------------------------------
# Grounding test: swap real context for irrelevant context
# --------------------------------------------------------------------------

def rag_generate_with_fake_context(
        query: str,
        fake_context: str,
        tokenizer,
        model,
        max_new_tokens: int = 512,
        do_sample: bool = True,
        temperature: float = 0.7) -> str:
    """Generate using irrelevant context instead of real recipes.

    Used for Task 7: verify the LM is actually using the provided context.
    If the model produces a sensible recipe answer from irrelevant context,
    it's relying on its own knowledge rather than the context.
    """
    import torch

    user_content = f"""{SYSTEM_A}

--- RECIPE CONTEXT ---
{fake_context}
--- END CONTEXT ---

Question: {query}

Answer the question based on the recipes above."""

    messages = [{"role": "user", "content": user_content}]
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=False
    ).to(model.device)

    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample)
    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        output_ids = model.generate(encoded, **gen_kwargs)

    generated_ids = output_ids[0, encoded.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [21]:
irrelevant_context = """
Richard Gary Brautigan (January 30, 1935 – c. September 16, 1984)
was an American novelist, poet, and short story writer. A prolific writer,
he wrote throughout his life and published ten novels, two collections of
short stories, and four books of poetry. Brautigan's work has been published
both in the United States and internationally throughout Europe, Japan,
and China. He is best known for his novels Trout Fishing in America (1967),
In Watermelon Sugar (1968), and The Abortion: An Historical Romance 1966 (1971).
"""

In [22]:
r = TfidfRetriever(fields=("name", "ingredients", "tags", "steps")).fit(dataset)
print(f"Retriever ready: {len(r.vectorizer.vocabulary_)} terms")

Retriever ready: 27642 terms


Before loading a model from the HuggingFace hub, you will likely want to create an account at https://huggingface.co/ so that you can get an [**access token**](https://huggingface.co/docs/hub/security-tokens) for models which require identification before usage.

- On your personal machine, you can input this access token by running `huggingface-cli login` in a terminal window.

- In Colab, click the icon in the left sidebar that looks like a key, and *Add a secret* called `HF_TOKEN`.

If you don't do this, you risk running into a "Cannot access gated repo" error.

In [23]:
# from google.colab import userdata
# userdata.get("HF_TOKEN")

**IMPORTANT**: only run the following code when you have implemented a working retrieval system. When you are ready to work with language models, navigate to the menu bar in Colab and select **Runtime > Change runtime type > T4 GPU**. If you find yourself working on not GPU-intenstive tasks in this notebook, change your runtime back to CPU to preserve access.


In [24]:
! pip -q install git+https://github.com/huggingface/transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.


In [25]:
! pip -q install datasets bitsandbytes accelerate xformers einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 21.5 MB/s eta 0:00:00


In [26]:
import torch
import transformers
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM

In [27]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

A tokenizer is required in order to convert strings into integer sequences that can be passed as input to the model.

In [28]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

The code below will load a Mistral 7B instruct model and quantize it via `bitesandbytes`. Doing so will ensure that the model will not take up too much memory and make inference more efficient. Note that the call to `AutoModelForCausalLM.from_pretrained()` will take a while, as the model's weights must be downloaded from the huggingface hub. Also note that you are not restricted to using Mistral, and are welcome to experiment with other models (though you will have more luck with chat and instruction-tuned variants).

In [29]:
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [30]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto'
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

We construct example input that could be passed to the model. Note that because we didn't specify any instructions in our prompt string, the model will behave as if it is just auto-completing whatever we gave it.

In [31]:
# ---- Task 6+7: RAG generation tests ----

# Test 1: Basic RAG with Prompt A (structured)
print("=" * 60)
print("TEST 1: Prompt A — structured")
print("=" * 60)
result_a = rag_generate(
    "What temperature should I pre-heat my oven to when making chicken quesadillas?",
    retriever=r, dataset=dataset, tokenizer=tokenizer, model=model,
    k=5, prompt_fn=build_prompt_a
)
print(f"\nQuery: {result_a['query']}")
print(f"\nRetrieved recipes:")
for h, rec in zip(result_a['hits'], result_a['recipes']):
    print(f"  {rec['name']} (score={h.score:.3f})")
print(f"\nResponse:\n{result_a['response']}")

# Test 2: Same query, Prompt B (minimal)
print("\n" + "=" * 60)
print("TEST 2: Prompt B — minimal")
print("=" * 60)
result_b = rag_generate(
    "What temperature should I pre-heat my oven to when making chicken quesadillas?",
    retriever=r, dataset=dataset, tokenizer=tokenizer, model=model,
    k=5, prompt_fn=build_prompt_b
)
print(f"\nResponse:\n{result_b['response']}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


TEST 1: Prompt A — structured


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Query: What temperature should I pre-heat my oven to when making chicken quesadillas?

Retrieved recipes:
  pepper jack chicken peach quesadillas (score=0.418)
  black bean chicken quesadillas (score=0.408)
  grilled quesadillas feta spinach olive lemon relish (score=0.375)
  apple pie quesadillas (score=0.373)
  ww weight watchers chicken cheese quesadillas (score=0.372)

Response:
none of the recipes provided call for pre-heating an oven to make chicken quesadillas. The instructions are to cook the quesadillas on a stovetop or grill. Therefore, there is no answer to this question based on the context given.

TEST 2: Prompt B — minimal

Response:
None of the provided recipes call for pre-heating the oven for making chicken quesadillas. The recipes instruct cooking the quesadillas on a stovetop or grill.


In [32]:
# encoded_prompt = tokenizer(input_string, return_tensors="pt", add_special_tokens=False)
# encoded_prompt = encoded_prompt.to("cuda")

In [33]:
# ---- Task 7: Grounding test + score inclusion test ----

# Grounding: does the LM answer from context or its own knowledge?
print("=" * 60)
print("GROUNDING TEST: Irrelevant context (Brautigan biography)")
print("=" * 60)
grounding_response = rag_generate_with_fake_context(
    "What temperature should I pre-heat my oven to when making chicken quesadillas?",
    fake_context=irrelevant_context,
    tokenizer=tokenizer, model=model
)
print(f"\nWith irrelevant context:\n{grounding_response}")
print(f"\nWith real context:\n{result_a['response']}")

# Score inclusion: does adding retrieval scores change the output?
print("\n" + "=" * 60)
print("SCORE INCLUSION TEST")
print("=" * 60)
result_with_scores = rag_generate(
    "What temperature should I pre-heat my oven to when making chicken quesadillas?",
    retriever=r, dataset=dataset, tokenizer=tokenizer, model=model,
    k=5, prompt_fn=build_prompt_a, include_scores=True
)
print(f"\nWith scores:\n{result_with_scores['response']}")
print(f"\nWithout scores:\n{result_a['response']}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


GROUNDING TEST: Irrelevant context (Brautigan biography)


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



With irrelevant context:
I cannot answer that question with the provided context as the context does not contain a recipe for chicken quesadillas.

With real context:
none of the recipes provided call for pre-heating an oven to make chicken quesadillas. The instructions are to cook the quesadillas on a stovetop or grill. Therefore, there is no answer to this question based on the context given.

SCORE INCLUSION TEST

With scores:
None of the provided recipes call for pre-heating the oven to make chicken quesadillas. The instructions for making quesadillas in all the recipes involve cooking them on a skillet or grill. Therefore, I cannot provide an answer to your question based on the given recipe context.

Without scores:
none of the recipes provided call for pre-heating an oven to make chicken quesadillas. The instructions are to cook the quesadillas on a stovetop or grill. Therefore, there is no answer to this question based on the context given.


In [34]:
# ---- Task 7: Does including retrieval scores change output? ----
print("=" * 60)
print("SCORE INCLUSION TEST")
print("=" * 60)

result_with_scores = rag_generate(
    "What temperature should I pre-heat my oven to when making chicken quesadillas?",
    retriever=r, dataset=dataset, tokenizer=tokenizer, model=model,
    k=5, prompt_fn=build_prompt_a, include_scores=True
)
print(f"\nWith scores:\n{result_with_scores['response']}")
print(f"\nWithout scores (from Test 1):\n{result_a['response']}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


SCORE INCLUSION TEST

With scores:
The recipes provided do not mention pre-heating the oven for making chicken quesadillas. Therefore, no answer can be given based on the information provided in the context.

Without scores (from Test 1):
none of the recipes provided call for pre-heating an oven to make chicken quesadillas. The instructions are to cook the quesadillas on a stovetop or grill. Therefore, there is no answer to this question based on the context given.


This is the final generation step, where a forward pass must be made through the entire model. Since the model is large (even after quantization), it might take a while.

In [35]:
# generated_ids = model.generate(**encoded_prompt, max_new_tokens=1000, do_sample=True)
# decoded = tokenizer.batch_decode(generated_ids)
# print(decoded[0])

We can see that, even without additional context and reference documents, the model is able to generate very coherent recipe instructions. Of course, asking the model to reason about a given set of documents won't work if it doesn't receive those documents.

For the questions we expect you to answer, refer to the assignment handout.

In [36]:
# ---- Task 7: 5 reasoning queries (cross-recipe reasoning) ----
# These must require the LM to reason ACROSS multiple retrieved recipes,
# not just return a recipe. "Recommend me a recipe" does NOT count.

reasoning_queries = [
    # 1. Comparison: which recipe is simpler?
    "Between chicken quesadillas and beef tacos, which requires fewer ingredients?",
    # 2. Aggregation: common pattern across recipes
    "What spice or seasoning appears most often across different chili recipes?",
    # 3. Substitution reasoning
    "If I'm allergic to dairy, which pasta recipes can I make without modification?",
    # 4. Contrast: different techniques for same goal
    "What are the different ways these recipes suggest cooking shrimp?",
    # 5. Inference: combining info from multiple docs
    "Based on these soup recipes, what is the typical cooking time for a homemade soup?",
]

for i, q in enumerate(reasoning_queries, 1):
    print(f"\n{'='*60}")
    print(f"REASONING QUERY {i}: {q}")
    print(f"{'='*60}")
    result = rag_generate(
        q, retriever=r, dataset=dataset, tokenizer=tokenizer, model=model,
        k=5, prompt_fn=build_prompt_a
    )
    print(f"\nRetrieved:")
    for h, rec in zip(result['hits'], result['recipes']):
        print(f"  {rec['name']} (score={h.score:.3f})")
    print(f"\nResponse:\n{result['response']}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



REASONING QUERY 1: Between chicken quesadillas and beef tacos, which requires fewer ingredients?


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Retrieved:
  pepper jack chicken peach quesadillas (score=0.272)
  black bean chicken quesadillas (score=0.266)
  quesadillas guacamole (score=0.256)
  shredded beef tacos burritos (score=0.249)
  grilled quesadillas feta spinach olive lemon relish (score=0.247)

Response:
Based on the provided recipes, the chicken quesadillas require more ingredients than the beef tacos. Here's a breakdown of the ingredients used in each recipe:

Chicken Quesadillas (Recipe 1): honey, fresh lime juice, reduced-fat sour cream, flour tortillas, cheese, cooked chicken breast, firm ripe peach, fresh cilantro, cooking spray
Beef Tacos (Recipe 4): beef roast, onion, beef broth, tomato sauce, lime juice, garlic cloves, cumin, chili powder, salt, cilantro, jalapeno pepper

From this list, it's clear that the chicken quesadillas require more ingredients than the beef tacos. While both recipes call for various spices and herbs, the chicken quesadillas also require honey, lime juice, sour cream, peaches, and ci

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Retrieved:
  kittencal chili seasoning mix (score=0.265)
  emeril spice blend recipes (score=0.264)
  chili spice seasoning (score=0.247)
  top secret recipes version red robin seasoning todd wilbur (score=0.237)
  panch phoron bengali five spice (score=0.230)

Response:
Based on the provided recipes, chili powder appears most often across different chili recipes. It is included in Recipe 1 (kittencal chili seasoning mix), Recipe 3 (chili spice seasoning), and Recipe 4 (top secret recipes version red robin seasoning todd wilbur). However, it's worth noting that there are other spices like paprika, cumin, dried oregano, garlic powder, and cayenne pepper that are also commonly used in chili recipes.

REASONING QUERY 3: If I'm allergic to dairy, which pasta recipes can I make without modification?


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Retrieved:
  seventies almond rice (score=0.213)
  cooked chicken recipes barefoot contessa style (score=0.157)
  yo naise light mayonnaise (score=0.156)
  turn regular pasta chinese oriental noodles (score=0.151)
  mom cake (score=0.150)

Response:
Based on the provided context, Recipe 1: Seventies Almond Rice and Recipe 4: Turn Regular Pasta Chinese Oriental Noodles do not contain any dairy ingredients in their original form. Therefore, you can make these recipes without any modification if you are allergic to dairy.

REASONING QUERY 4: What are the different ways these recipes suggest cooking shrimp?


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Retrieved:
  barbecued recipes grilled shrimp (score=0.285)
  curry paste shrimp (score=0.273)
  garlic orange shrimp (score=0.263)
  adobo style shrimp filipino (score=0.257)
  boiled shrimp salads (score=0.255)

Response:
The recipes suggest the following ways to cook shrimp:

1. Barbecued shrimp: Marinate shrimp in a mixture of ingredients and grill on a wire grill basket or skewers.
2. Curry paste shrimp: If using raw shrimp, stir-fry in a non-stick skillet until they turn bright coral in color. If using frozen cooked shrimp, stir-fry for just 2 minutes until heated through.
3. Garlic orange shrimp: Marinate shrimp in garlic, red chili pepper, olive oil, and parsley, then cook in a frying pan.
4. Adobo style shrimp: Cook shrimp in a pan or wok with vinegar, water, soy sauce, garlic cloves, salt, pepper, and sugar. Then, fry shrimp and garlic together in cooking oil.
5. Boiled shrimp for salads: Boil shrimp in a large saucepan with water, salt, bay leaf, lemon juice, rice vinegar, 